# 02 — Cruzamento da avaliação humana com a chave de amostragem

Este notebook deve ser executado **depois que `avaliacao_humana.json` estiver preenchido**.

Ele:

1. lê `avaliacao_humana.json`;
2. lê `chave_amostragem.json`;
3. cruza os arquivos por `avaliacao_id`;
4. valida se `case_id`, caso original e BDD avaliado continuam corretos;
5. valida notas entre 0 e 10;
6. calcula:

\[
\text{Nota Final} =
0{,}4 \times \text{Estrutura}
+
0{,}4 \times \text{Semântica}
+
0{,}2 \times \text{Detalhes}
\]

7. recupera exatamente:
   - modelo;
   - técnica;
   - execução;
   - `generation_id`.

O campo **`generation_id` é preservado como chave canônica para o futuro cruzamento com METEOR, Manhattan, NLI etc.**  
Assim, a métrica utilizada será sempre a da **mesma execução que foi avaliada pelo humano**.

## Saída

`avaliacao_humana_consolidada.json`


In [ ]:
from pathlib import Path
from collections import Counter
import json
import math

## 1. Configurações

In [ ]:
ARQUIVO_AVALIACAO = Path("avaliacao_humana.json")
ARQUIVO_CHAVE = Path("chave_amostragem.json")

ARQUIVO_SAIDA = Path("avaliacao_humana_consolidada.json")

NOTA_MINIMA = 0
NOTA_MAXIMA = 10

# Se True, também verifica se o avaliador não alterou
# acidentalmente o texto original ou o BDD sorteado.
VALIDAR_TEXTOS = True

## 2. Funções auxiliares

In [ ]:
def carregar_json(caminho):
    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho.resolve()}"
        )

    with open(caminho, "r", encoding="utf-8") as arquivo:
        return json.load(arquivo)


def salvar_json(caminho, dados):
    with open(caminho, "w", encoding="utf-8") as arquivo:
        json.dump(
            dados,
            arquivo,
            ensure_ascii=False,
            indent=2
        )


def normalizar_texto(texto):
    if texto is None:
        return None

    # Mantém o conteúdo, mas neutraliza diferenças puramente
    # relacionadas a espaços/quebras de linha.
    return " ".join(str(texto).split())


def validar_nota(valor, criterio, avaliacao_id):
    if valor is None:
        raise ValueError(
            f"{avaliacao_id}: nota '{criterio}' não preenchida."
        )

    if isinstance(valor, bool) or not isinstance(valor, (int, float)):
        raise ValueError(
            f"{avaliacao_id}: nota '{criterio}' deve ser numérica. "
            f"Valor encontrado: {valor!r}"
        )

    if not math.isfinite(float(valor)):
        raise ValueError(
            f"{avaliacao_id}: nota '{criterio}' não é finita."
        )

    if not (NOTA_MINIMA <= float(valor) <= NOTA_MAXIMA):
        raise ValueError(
            f"{avaliacao_id}: nota '{criterio}' fora do intervalo "
            f"{NOTA_MINIMA}-{NOTA_MAXIMA}. "
            f"Valor encontrado: {valor}"
        )

    return float(valor)


def calcular_nota_final(estrutura, semantica, detalhes):
    nota = (
        estrutura * 0.4
        + semantica * 0.4
        + detalhes * 0.2
    )

    return round(nota, 2)

## 3. Carregar os dois JSONs

In [ ]:
avaliacao_humana = carregar_json(
    ARQUIVO_AVALIACAO
)

chave_amostragem = carregar_json(
    ARQUIVO_CHAVE
)

if "avaliacoes" not in avaliacao_humana:
    raise ValueError(
        "avaliacao_humana.json não possui a chave 'avaliacoes'."
    )

if "selecoes" not in chave_amostragem:
    raise ValueError(
        "chave_amostragem.json não possui a chave 'selecoes'."
    )

avaliacoes = avaliacao_humana["avaliacoes"]
selecoes = chave_amostragem["selecoes"]

print(f"Avaliações humanas: {len(avaliacoes)}")
print(f"Registros da chave: {len(selecoes)}")

Avaliações humanas: 259
Registros da chave: 259


## 4. Criar índice da chave por `avaliacao_id`

In [ ]:
indice_chave = {}

for selecao in selecoes:
    avaliacao_id = selecao.get("avaliacao_id")

    if not avaliacao_id:
        raise ValueError(
            "Existe registro na chave sem avaliacao_id."
        )

    if avaliacao_id in indice_chave:
        raise ValueError(
            f"avaliacao_id duplicado na chave: {avaliacao_id}"
        )

    indice_chave[avaliacao_id] = selecao

print(f"IDs indexados: {len(indice_chave)}")

IDs indexados: 259


## 5. Validar correspondência entre os arquivos

In [ ]:
ids_avaliacao = []

for avaliacao in avaliacoes:
    avaliacao_id = avaliacao.get("avaliacao_id")

    if not avaliacao_id:
        raise ValueError(
            "Existe avaliação humana sem avaliacao_id."
        )

    ids_avaliacao.append(avaliacao_id)

if len(ids_avaliacao) != len(set(ids_avaliacao)):
    repetidos = [
        item
        for item, quantidade in Counter(ids_avaliacao).items()
        if quantidade > 1
    ]

    raise ValueError(
        f"avaliacao_id duplicado na avaliação humana: {repetidos}"
    )

ids_avaliacao = set(ids_avaliacao)
ids_chave = set(indice_chave.keys())

faltando_na_chave = sorted(ids_avaliacao - ids_chave)
faltando_na_avaliacao = sorted(ids_chave - ids_avaliacao)

if faltando_na_chave:
    raise ValueError(
        f"Avaliações sem correspondência na chave: {faltando_na_chave}"
    )

if faltando_na_avaliacao:
    raise ValueError(
        f"Registros da chave sem avaliação humana: {faltando_na_avaliacao}"
    )

print("Todos os avaliacao_id possuem correspondência 1:1.")

Todos os avaliacao_id possuem correspondência 1:1.


## 6. Cruzar, validar notas e calcular nota final

In [ ]:
resultados = []

for avaliacao in avaliacoes:
    avaliacao_id = avaliacao["avaliacao_id"]
    chave = indice_chave[avaliacao_id]

    # --------------------------------------------------------
    # Validar o caso
    # --------------------------------------------------------
    case_id_avaliacao = avaliacao.get("case_id")
    case_id_chave = chave.get("case_id")

    if case_id_avaliacao != case_id_chave:
        raise ValueError(
            f"{avaliacao_id}: case_id divergente. "
            f"Avaliação={case_id_avaliacao}, chave={case_id_chave}"
        )

    # --------------------------------------------------------
    # Validar textos, se habilitado
    # --------------------------------------------------------
    if VALIDAR_TEXTOS:
        original_avaliacao = normalizar_texto(
            avaliacao.get("caso_original")
        )

        original_chave = normalizar_texto(
            chave.get("original_case")
        )

        if original_avaliacao != original_chave:
            raise ValueError(
                f"{avaliacao_id}: o caso original foi alterado "
                f"entre a chave e a avaliação."
            )

        bdd_avaliacao = normalizar_texto(
            avaliacao.get("bdd_gerado")
        )

        bdd_chave = normalizar_texto(
            chave.get("gherkin")
        )

        if bdd_avaliacao != bdd_chave:
            raise ValueError(
                f"{avaliacao_id}: o BDD avaliado não corresponde "
                f"ao BDD sorteado na chave."
            )

    # --------------------------------------------------------
    # Ler e validar notas
    # --------------------------------------------------------
    notas = avaliacao.get("avaliacao", {})

    estrutura = validar_nota(
        notas.get("estrutura"),
        "estrutura",
        avaliacao_id
    )

    semantica = validar_nota(
        notas.get("semantica"),
        "semantica",
        avaliacao_id
    )

    detalhes = validar_nota(
        notas.get("detalhes"),
        "detalhes",
        avaliacao_id
    )

    nota_final = calcular_nota_final(
        estrutura,
        semantica,
        detalhes
    )

    # --------------------------------------------------------
    # Registro consolidado
    #
    # generation_id é a chave que deverá ser usada mais tarde
    # para recuperar exatamente as métricas da mesma execução.
    # --------------------------------------------------------
    registro = {
        "avaliacao_id": avaliacao_id,
        "case_id": chave["case_id"],
        "source_id": chave.get("source_id"),
        "source_line": chave.get("source_line"),

        "caso_original": chave.get("original_case"),
        "bdd_gerado": chave.get("gherkin"),

        "model": chave.get("model"),
        "technique": chave.get("technique"),
        "execution": int(chave.get("execution")),
        "generation_id": chave.get("generation_id"),

        "estrutura": estrutura,
        "semantica": semantica,
        "detalhes": detalhes,
        "nota_final": nota_final
    }

    resultados.append(registro)

print(f"Registros consolidados: {len(resultados)}")

Registros consolidados: 259


## 7. Validar `generation_id` e a execução exata

In [ ]:
generation_ids = [
    item["generation_id"]
    for item in resultados
]

if any(not generation_id for generation_id in generation_ids):
    raise ValueError(
        "Existe registro consolidado sem generation_id."
    )

if len(generation_ids) != len(set(generation_ids)):
    repetidos = [
        item
        for item, quantidade in Counter(generation_ids).items()
        if quantidade > 1
    ]

    raise ValueError(
        f"generation_id duplicado: {repetidos}"
    )

# Validação adicional:
# o sufixo da execution deve ser compatível com o generation_id.
for item in resultados:
    sufixo_esperado = f"__exec-{item['execution']:02d}"

    if not item["generation_id"].endswith(sufixo_esperado):
        raise ValueError(
            f"{item['avaliacao_id']}: generation_id não corresponde "
            f"à execução {item['execution']}. "
            f"generation_id={item['generation_id']}"
        )

print(
    "Todos os generation_id são únicos e correspondem "
    "à execução registrada."
)

Todos os generation_id são únicos e correspondem à execução registrada.


## 8. Gerar JSON consolidado

In [ ]:
saida = {
    "metadata": {
        "total_avaliacoes": len(resultados),
        "escala": {
            "minimo": NOTA_MINIMA,
            "maximo": NOTA_MAXIMA
        },
        "pesos": {
            "estrutura": 0.4,
            "semantica": 0.4,
            "detalhes": 0.2
        },
        "formula_nota_final": (
            "(estrutura * 0.4) + "
            "(semantica * 0.4) + "
            "(detalhes * 0.2)"
        ),
        "chave_para_cruzamento_com_metricas": "generation_id",
        "observacao_metricas": (
            "O generation_id identifica exatamente o caso, modelo, "
            "tecnica e execucao avaliados pelo humano."
        ),
        "arquivos_origem": {
            "avaliacao_humana": str(ARQUIVO_AVALIACAO),
            "chave_amostragem": str(ARQUIVO_CHAVE)
        }
    },
    "resultados": resultados
}

salvar_json(
    ARQUIVO_SAIDA,
    saida
)

print(f"Gerado: {ARQUIVO_SAIDA.resolve()}")

Gerado: /content/avaliacao_humana_consolidada.json


## 9. Resumo das notas e das condições experimentais

In [ ]:
notas_finais = [
    item["nota_final"]
    for item in resultados
]

media = sum(notas_finais) / len(notas_finais)
menor = min(notas_finais)
maior = max(notas_finais)

print("=" * 70)
print("CONSOLIDAÇÃO CONCLUÍDA")
print("=" * 70)

print(f"Total: {len(resultados)}")
print(f"Média da nota final: {media:.2f}")
print(f"Menor nota: {menor:.2f}")
print(f"Maior nota: {maior:.2f}")

print("\nPor modelo:")
for modelo, quantidade in sorted(
    Counter(item["model"] for item in resultados).items()
):
    print(f"  {modelo}: {quantidade}")

print("\nPor técnica:")
for tecnica, quantidade in sorted(
    Counter(item["technique"] for item in resultados).items()
):
    print(f"  {tecnica}: {quantidade}")

print("\nPor execução:")
for execucao in range(1, 11):
    quantidade = sum(
        1
        for item in resultados
        if item["execution"] == execucao
    )
    print(f"  Execução {execucao}: {quantidade}")

print(f"\nArquivo final: {ARQUIVO_SAIDA}")

CONSOLIDAÇÃO CONCLUÍDA
Total: 259
Média da nota final: 6.93
Menor nota: 3.80
Maior nota: 8.50

Por modelo:
  Qwen/Qwen3-8B: 52
  google/gemma-4-E4B-it: 52
  ibm-granite/granite-4.1-8b: 51
  meta-llama/Meta-Llama-3-8B-Instruct: 52
  mistralai/Mistral-7B-Instruct-v0.3: 52

Por técnica:
  few-shot: 86
  one-shot: 87
  zero-shot: 86

Por execução:
  Execução 1: 26
  Execução 2: 26
  Execução 3: 26
  Execução 4: 26
  Execução 5: 26
  Execução 6: 26
  Execução 7: 26
  Execução 8: 25
  Execução 9: 26
  Execução 10: 26

Arquivo final: avaliacao_humana_consolidada.json


## 10. Próxima etapa: métricas automáticas

O arquivo `avaliacao_humana_consolidada.json` já fica preparado para a correlação.

A chave recomendada para cruzar com os arquivos de métricas é:

```text
generation_id
```

Exemplo:

```text
TC_177__google-gemma-4-e4b-it__one-shot__exec-07
```

Essa chave impede que a nota humana da execução 7 seja associada, por engano, à métrica das execuções 1, 2, 3 etc.

Como validação redundante, na futura etapa de métricas também é recomendável confirmar:

- `case_id`
- `model` / `modelo`
- `technique` / `tecnica`
- `execution` / `execucao`
